# 02 · Votação de features

**Features F3 e F4** (critérios 3 e 4) · donos: ver `config/equipe.yaml` · **status: F3 pronta; F4 a implementar**

> Duas pessoas, um notebook: **F3 edita só a seção F3** e **F4 só a seção F4**, para evitar conflito de merge. Cada seção chama o módulo da própria feature (`src/features.py` e `src/votacao.py`).

| | |
|---|---|
| **Entradas** | `ciclo_atual.json`, `dev.parquet`, `cfg['selecao']` |
| **Saídas** | `data/processed/features_eleitas.json` · `evidencias/selecao/rankings_<cycle_id>.csv` · `evidencias/votacao/votos_<cycle_id>.csv` · registro no MLflow |

**Regras do enunciado que se aplicam**
- Três métodos no mesmo notebook (SelectKBest, importância do Random Forest e RFE, ou três modelos com feature importance).
- Regra de votação definida **antes** (está em `config.yaml`); registrar tabela de votos, desempate e mínimo de features.
- Ajustar só no treino; com validação cruzada, dentro de cada fold.

Contrato completo na docstring do módulo `src/` correspondente.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))

from src.config import carregar_config, carregar_contexto
from src.preprocessamento import carregar_particao, construir_preprocessador, lista_features

cfg = carregar_config()
ctx = carregar_contexto(cfg, RAIZ)  # escrito pelo notebook 01
print(f"cycle_id={ctx['cycle_id']} | data_version={ctx['data_version']}")

cycle_id=20261105T113604Z-local | data_version=dv018-f95c3940


## Seção F3 · três métodos de seleção (`src/features.py`)

In [2]:
from IPython.display import display

from src.features import calcular_rankings_por_fold, salvar_rankings, tabela_rankings

dev = carregar_particao("dev", cfg, RAIZ)
assert dev[ctx["colunas"]["tempo"]].is_monotonic_increasing, "dev fora de ordem: o TimeSeriesSplit vazaria o futuro"
X_bruto = dev[lista_features(cfg)]
y = dev[ctx["colunas"]["alvo"]]
k_top = cfg["selecao"]["k_top"]
print(f"dev: {len(dev):,} linhas | {X_bruto.shape[1]} colunas antes do pré-processador | k_top={k_top}")
print("métodos:", cfg["selecao"]["metodos"])

# Pré-processador e métodos são reajustados só com o treino de cada fold.
rankings, por_fold = calcular_rankings_por_fold(X_bruto, y, cfg)
tabela = tabela_rankings(rankings, por_fold)
print(f"{len(tabela)} features após o pré-processador (união dos folds)")

dev: 62,807 linhas | 32 colunas antes do pré-processador | k_top=20
métodos: {'selectkbest': {'score_func': 'mutual_info_classif'}, 'random_forest': {'n_estimators': 200, 'max_depth': 12, 'n_jobs': -1}, 'rfe': {'estimador': 'logistic_regression', 'step': 0.1, 'params': {'max_iter': 1000}}}


89 features após o pré-processador (união dos folds)


### Top-k de cada método
Score médio entre os folds. As escalas diferem entre métodos, então só a ordem dentro de cada coluna é comparável.

In [3]:
lado_a_lado = {}
for metodo, score in rankings.items():
    top = score.sort_values(ascending=False, kind="mergesort").head(k_top)
    lado_a_lado[(metodo, "feature")] = top.index.to_list()
    lado_a_lado[(metodo, "score")] = top.round(4).to_list()
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)
display(pd.DataFrame(lado_a_lado, index=range(1, k_top + 1)))

selectkbest                          random_forest                                  rfe        
                        feature   score                        feature   score                      feature   score
1           razao_cancel_previo  0.1994            razao_cancel_previo  0.1354                    agent_240  1.0000
2        previous_cancellations  0.1903         previous_cancellations  0.1254                      agent_9  1.0000
3       deposit_type_Non Refund  0.1761        deposit_type_Non Refund  0.1228                  country_ITA  1.0000
4       deposit_type_No Deposit  0.1718        deposit_type_No Deposit  0.1200                  country_PRT  1.0000
5                     lead_time  0.1686                    country_PRT  0.1063      deposit_type_No Deposit  1.0000
6        valor_estimado_estadia  0.1592                      lead_time  0.0699      deposit_type_Non Refund  1.0000
7                           adr  0.1408                            adr  0.0217          razao_cancel_previo  1.0000
8                   country_PRT  0.1214      total_of_special_requests  0.0192  required_car_parking_spaces  1.0000
9         market_segment_Groups  0.0564         valor_estimado_estadia  0.0189       customer_type_Contract  0.9500
10                      agent_1  0.0458          market_segment_Groups  0.0187                  tem_empresa  0.9500
11    total_of_special_requests  0.0334    required_car_parking_spaces  0.0163                  country_DEU  0.9250
12  required_car_parking_spaces  0.0309       arrival_date_week_number  0.0148                  country_GBR  0.9167
13   country_infrequent_sklearn  0.0243                        agent_1  0.0130                  country_IRL  0.9167
14         days_in_waiting_list  0.0213                    mes_chegada  0.0116        market_segment_Groups  0.8889
15                 total_noites  0.0198        customer_type_Transient  0.0109                      meal_FB  0.8750
16                  country_ESP  0.0186                   total_noites  0.0109  distribution_channel_Direct  0.8444
17         stays_in_week_nights  0.0179     country_infrequent_sklearn  0.0109            is_repeated_guest  0.8250
18                  country_FRA  0.0174                        agent_9  0.0101                     agent_14  0.8000
19             hotel_City Hotel  0.0165  customer_type_Transient-Party  0.0095                      meal_SC  0.7583
20           hotel_Resort Hotel  0.0165                    country_ESP  0.0082                      agent_1  0.7556

### Tabela completa de scores por método
Mesmo índice nos três métodos. `posicao` 1 é a melhor; `folds_presente` diz em quantos folds o one-hot gerou a coluna.

In [4]:
display(tabela.sort_values("posicao_random_forest").round(4))

,score_selectkbest,posicao_selectkbest,score_random_forest,posicao_random_forest,score_rfe,posicao_rfe,folds_presente
feature,,,,,,,
razao_cancel_previo,0.1994,1,0.1354,1,1.0000,1,4
previous_cancellations,0.1903,2,0.1254,2,0.4139,60,4
deposit_type_Non Refund,0.1761,3,0.1228,3,1.0000,1,4
deposit_type_No Deposit,0.1718,4,0.1200,4,1.0000,1,4
country_PRT,0.1214,8,0.1063,5,1.0000,1,4
lead_time,0.1686,5,0.0699,6,0.5833,34,4
adr,0.1408,7,0.0217,7,0.3472,67,4
total_of_special_requests,0.0334,11,0.0192,8,0.5917,32,4
valor_estimado_estadia,0.1592,6,0.0189,9,0.5750,36,4


### Concordância entre os métodos e estabilidade entre folds
Os métodos só agregam na votação se não forem cópias um do outro, e um ranking que muda muito entre folds merece desconfiança.

In [5]:
tops = {m: set(tabela.nsmallest(k_top, f"posicao_{m}").index) for m in rankings}
display(pd.DataFrame(
    {a: {b: len(tops[a] & tops[b]) for b in tops} for a in tops}
).rename_axis("features em comum no top-k"))

estabilidade = {}
for metodo, grupo in por_fold.groupby("metodo"):
    tops_fold = [set(g.nlargest(k_top, "score")["feature"]) for _, g in grupo.groupby("fold")]
    estabilidade[metodo] = {
        f"folds {i + 1}-{i + 2}": len(a & b) / len(a | b) for i, (a, b) in enumerate(zip(tops_fold, tops_fold[1:]))
    }
display(pd.DataFrame(estabilidade).T.round(2).rename_axis("Jaccard do top-k entre folds seguidos"))

em_todos = set.union(*tops.values())
parciais = tabela.loc[sorted(em_todos), "folds_presente"]
print("no top-k de algum método, mas ausentes em algum fold:", parciais[parciais < cfg["avaliacao"]["cv"]["n_splits"]].to_dict())

,selectkbest,random_forest,rfe
features em comum no top-k,,,
selectkbest,20,15,7
random_forest,15,20,8
rfe,7,8,20


,folds 1-2,folds 2-3,folds 3-4
Jaccard do top-k entre folds seguidos,,,
random_forest,0.82,0.82,1.00
rfe,0.38,0.54,0.67
selectkbest,0.82,0.82,1.00


no top-k de algum método, mas ausentes em algum fold: {}


In [6]:
caminhos_rankings = salvar_rankings(tabela, por_fold, ctx, RAIZ)
for nome, caminho in caminhos_rankings.items():
    print(f"{nome}: {caminho.relative_to(RAIZ)}")

rankings: evidencias/selecao/rankings_20261105T113604Z-local.csv
por_fold: evidencias/selecao/rankings_por_fold_20261105T113604Z-local.csv


### Limitações da seleção
- **Cada método tem um viés.** A informação mútua é univariada e ignora redundância. A importância por impureza favorece variáveis contínuas e de muitos valores. O RFE depende dos coeficientes de um modelo linear, e dummies não têm a mesma escala das numéricas padronizadas. É por isso que a votação da F4 combina os três.
- **Atributos redundantes dividem o crédito.** `tem_agente` e `agent_SEM_AGENTE` são exatamente complementares, e `total_noites` é a soma de outras duas colunas. O RFE pode manter uma e descartar a outra sem que a descartada seja inútil.
- **Média entre folds não é seleção por fold.** O treino dos folds finais inclui a validação dos anteriores, então o CV da F5 com um único conjunto eleito sai levemente otimista. Só a validação do último fold e o teste ficam intocados pela seleção. `rankings_por_fold_<cycle_id>.csv` guarda cada fold, caso a F5 queira medir esse efeito.
- **`country` pode vazar o desfecho.** O artigo do dataset avisa que os hotéis costumam só conhecer a nacionalidade correta no check-in, que reservas canceladas não chegam a ter. Isso conflita com a previsão no ato da reserva definida na Etapa 1. Se `country_*` aparecer no topo, trate como suspeita, não como descoberta. Medição em `evidencias/selecao/LEIAME.md`.

## Seção F4 · votação e eleição (`src/votacao.py`)

In [7]:
# TODO(F4): ver src/votacao.py
# tabela = votar(rankings, cfg)
# eleitas = eleger_features(tabela, cfg)
# gravar data/processed/features_eleitas.json e evidencias/votacao/votos_<cycle_id>.csv
raise NotImplementedError("Seção F4 ainda não implementada")

NotImplementedError: Seção F4 ainda não implementada